# Step 3. RAG: 문서를 찾아서 LLM에게 함께 보여 주기
LLM은 우리 병원의 안내문을 모릅니다. 모르면 지어냅니다. **RAG(래그, Retrieval-Augmented Generation)** 는 답하기 전에 관련 문서를 찾아서 함께 보여 주는 방법입니다. 오픈북 시험과 같습니다.

## 순서
1. 문서를 작은 조각(Chunk)으로 쪼개서 Vector DB에 저장
2. 질문과 가까운 조각을 찾기 (Retrieval)
3. 질문과 찾은 조각을 함께 LLM에 넣기 (Generation)

> 아래 안내문은 실습용으로 지어낸 가상의 병원입니다.

In [ ]:
%pip install -q --disable-pip-version-check ollama chromadb

## 1. 설정

In [ ]:
import json

try:
    with open("../config.json") as f:
        _config = json.load(f)
    print("✅ config.json 로드 완료")
except FileNotFoundError:
    raise SystemExit("❌ config.json이 없습니다. Step 0 노트북을 먼저 실행해 주세요.")

EMBED_MODEL = _config["EMBED_MODEL"]
LLM_MODEL = _config["LLM_MODEL"]
AGENT_MODEL = _config["AGENT_MODEL"]
print(f"   Embedding 모델: {EMBED_MODEL}")
print(f"   LLM 모델:       {LLM_MODEL}")
print(f"   Agent 모델:     {AGENT_MODEL}")

## 2. 문서 준비와 쪼개기
빈 줄을 기준으로 문단마다 한 조각으로 나눕니다. 조각마다 주제가 하나여야 검색이 잘 됩니다.

In [ ]:
# TODO: 내 문서로 바꿔 보세요!
DOCUMENT = """
[진료 시간] 성균가상병원 외래 진료는 평일 오전 9시부터 오후 5시 30분까지입니다. 토요일은 오전 9시부터 12시 30분까지이며 일요일과 공휴일은 휴진입니다.

[영상 검사] 흉부 엑스레이는 예약 없이 본관 2층 영상의학과에서 바로 찍을 수 있습니다. CT와 MRI는 예약이 필요하며 검사 6시간 전부터 금식해야 합니다.

[주차] 주차장은 본관 지하 2층과 지하 3층입니다. 외래 환자는 4시간, 입원 환자 보호자는 하루 1대에 한해 무료입니다.

[서류 발급] 진단서와 진료 기록 사본은 본관 1층 원무과에서 발급합니다. 본인 신분증이 필요하며, 대리인은 위임장과 가족관계증명서를 가져와야 합니다.

[면회] 입원 환자 면회는 평일 오후 6시부터 8시까지, 주말은 오전 10시부터 12시까지 가능합니다. 중환자실은 하루 두 번 30분씩만 가능합니다.
"""

chunks = [c.strip() for c in DOCUMENT.strip().split("\n\n") if c.strip()]
print(f"✅ 문서를 {len(chunks)}개 조각으로 나눴습니다.")
for i, c in enumerate(chunks):
    print(f"   {i}: {c[:40]}...")

## 3. Vector DB에 저장

In [ ]:
import chromadb
import ollama

client = chromadb.PersistentClient(path="../.lab-data/step3")
col = client.get_or_create_collection("hospital", metadata={"hnsw:space": "cosine"})


def embed(text):
    return ollama.embed(model=EMBED_MODEL, input=text)["embeddings"][0]


col.upsert(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=[embed(c) for c in chunks],
)
print(f"✅ 저장 완료. 조각 수: {col.count()}개")

## 4. RAG 없이 물어보기
먼저 문서 없이 물어봅니다. LLM은 이 병원을 모르기 때문에 모른다고 하거나, 그럴듯하게 지어냅니다.

In [ ]:
def ask_llm(prompt):
    r = ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}], options={"temperature": 0.2, "num_predict": 300})
    return r["message"]["content"].strip()


question = "성균가상병원에서 CT 찍으려면 금식해야 해? 몇 시간?"
print("👤 질문:", question)
print("🤖 RAG 없이:", ask_llm(question))

## 5. RAG로 물어보기
가까운 조각 2개를 찾아서 질문 앞에 붙입니다. "문서에 없으면 모른다고 답하라"는 지시가 Guardrail(가드레일) 역할을 합니다.

In [ ]:
def ask_rag(question, k=2):
    r = col.query(query_embeddings=[embed(question)], n_results=k)
    found = r["documents"][0]
    context = "\n".join(f"- {d}" for d in found)
    prompt = f"""아래 [문서]만 보고 [질문]에 한국어로 짧게 답하세요.
문서에 없는 내용이면 "문서에 없습니다"라고 답하세요.

[문서]
{context}

[질문]
{question}"""
    print("👤 질문:", question)
    print("📄 찾은 조각:")
    for d, dist in zip(found, r["distances"][0]):
        print(f"   거리 {dist:.3f}  {d[:50]}...")
    print("🤖 RAG 답:", ask_llm(prompt))
    print()


ask_rag(question)

### 직접 해 보기

In [ ]:
# TODO: 질문을 바꿔 보세요!
ask_rag("토요일에도 진료해? 몇 시까지?")
ask_rag("아버지 진단서를 내가 대신 받으려면 뭐가 필요해?")
ask_rag("병원 근처 맛집 알려 줘")   # 문서에 없는 질문

## 정리
- RAG는 "찾기 + 함께 읽고 답하기"입니다. 문서가 바뀌면 DB만 고치면 됩니다.
- 지금까지는 **우리가 코드로** 검색을 호출했습니다. 다음은 **LLM이 스스로** 검색 도구를 고르게 합니다. → Step 4 (MCP)